# Israeli legal eval v2: check the gold against the corpus (CPU, ~15 min)

`legal_txt/Evals/israeli_legal_eval_v2` has 770 items: the 500 of v1 (rebalanced, split into test/dev/reserve),
195 new items (negated yes/no, comparison, multi-hop, computation, versioned law, abstention, application,
issue spotting) and 75 paraphrases. The new items were written without the statute text, so this notebook
checks every gold citation against the corpus before any score counts:

- **verify**: finds each cited law and section in the corpus, records the chunk ids as `gold_chunks`, and checks
  that the numbers a rule states (periods, amounts, ages) appear in the cited text;
- **select**: writes the test, dev and robustness splits, with and without the unverified new items.

No GPU and no model are needed. Send `eval_v2_verified.zip` back: the items it flags go to a lawyer (or back to
Claude) before the first v2 run.

## Before running
1. **Add-ons -> Secrets**: `GITHUB_TOKEN` with read access to `zananiri/AI-IZ`.
2. **Add Input**: the corpus dataset (`legal_corpus_vectordb`), the same as the eval runs.
3. Settings -> Accelerator: **None** (CPU is enough). Internet: **on**.
4. **Run All**.

In [ ]:
BRANCH = "main"   # the branch that has legal_txt/Evals/israeli_legal_eval_v2
OUT = "data/legal/eval_v2"


In [ ]:
import os, pathlib, shutil, subprocess, json
from kaggle_secrets import UserSecretsClient

def sh(command, check=True):
    proc = subprocess.run(["bash", "-o", "pipefail", "-c", command], text=True)
    if proc.returncode and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
sh("git log --oneline -1")
sh('pip install -q uv && uv pip install -q --system -e ".[legal]" pytest')
sh("python -m pytest -q -p no:cacheprovider tests/unit/test_legal_eval_v2.py")


In [ ]:
# The corpus database: category folders (laws/, procedural_rules/) that hold chroma.sqlite3, found
# anywhere under /kaggle/input -- unpacked (Kaggle unzips uploads) or still zipped, at any folder depth.
# Folders of .jsonl records (the build's legal_corpus_jsonl) have the same category names but no
# chroma.sqlite3: they are ignored. The 28 Sept run copied that folder instead and ran without a corpus.
import zipfile
INPUT = pathlib.Path("/kaggle/input")
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
CATEGORIES_WANTED = ("laws", "procedural_rules")
db_roots = sorted({p.parent.parent for p in INPUT.rglob("chroma.sqlite3") if p.parent.name in CATEGORIES_WANTED})
print("database folders found:", [str(r) for r in db_roots] or "none")
shutil.rmtree(VECTORDB, ignore_errors=True)
VECTORDB.mkdir(parents=True)
if db_roots:
    for item in db_roots[0].iterdir():
        (shutil.copytree if item.is_dir() else shutil.copy2)(item, VECTORDB / item.name)
    print("copied", db_roots[0], "->", VECTORDB)
else:
    for z in sorted(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(z) as archive:
            dbs = [n for n in archive.namelist()
                   if n.endswith("chroma.sqlite3") and len(n.split("/")) >= 2 and n.split("/")[-2] in CATEGORIES_WANTED]
            if not dbs:
                continue
            parts = dbs[0].split("/")  # [..., <category>, "chroma.sqlite3"]
            prefix = "/".join(parts[:-2]) + "/" if len(parts) > 2 else ""
            for member in archive.infolist():
                name = member.filename
                if name.startswith(prefix) and not name.endswith("/") and ".." not in name:
                    dest = VECTORDB / name[len(prefix):]
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as fsrc, open(dest, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst)
            print("unpacked", z, "->", VECTORDB)
            break

CORPUS_CATEGORIES = sorted(c for c in CATEGORIES_WANTED if (VECTORDB / c / "chroma.sqlite3").exists())
assert CORPUS_CATEGORIES, ("NO CORPUS DATABASE FOUND under /kaggle/input -- stopping instead of answering without it. "
                           "Attach the dataset with legal_corpus_vectordb (its laws/ and procedural_rules/ folders "
                           "must contain chroma.sqlite3).")
missing = [c for c in CATEGORIES_WANTED if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")


In [ ]:
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
cats = ",".join(CORPUS_CATEGORIES)
sh(f"python scripts/legal_data/eval_v2.py verify --gold legal_txt/Evals/israeli_legal_eval_v2/gold.jsonl "
   f"--vectordb-dir {VECTORDB} --categories {cats} --out {OUT}/gold_verified.jsonl --report {OUT}/verification.md")
for split in ("test", "dev", "robustness"):
    sh(f"python scripts/legal_data/eval_v2.py select --gold {OUT}/gold_verified.jsonl --split {split} --out-dir {OUT}/{split}")
    sh(f"python scripts/legal_data/eval_v2.py select --gold {OUT}/gold_verified.jsonl --split {split} "
       f"--out-dir {OUT}/{split}_verified_only --verified-only")


In [ ]:
DEST = "/kaggle/working/eval_v2_verified.zip"
sh(f"cd /kaggle/working && zip -qr {DEST} AI-IZ/{OUT}")
print("results zip:", DEST, f"({pathlib.Path(DEST).stat().st_size / 1e6:.1f} MB)")
